<a href="https://colab.research.google.com/github/mnsbharadwaj/AI-NLP/blob/master/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**RAG : Retrieval-Augmented Generation**

RAG (Retrieval-Augmented Generation) is a powerful method that combines retrieval-based and generation-based techniques for answering questions or generating text. The core idea behind RAG is to retrieve relevant documents (contexts) and then use them to generate accurate, grounded, and fluent responses.

**Core Components of RAG:**

RAG consists of two major components:


**Retriever (Dense Retriever)** : Retrieves top-k relevant documents using embeddings.

**Generator** (e.g., BART, T5):	Generates final answer based on retrieved documents.


**Retriever (Dense Passage Retriever - DPR)**

  Purpose:
Convert input queries and documents into dense vectors (embeddings) and retrieve the top-k similar documents.

Key Modules:

Context Encoder: Embeds documents.

Question Encoder: Embeds query.

Similarity Search: Typically via FAISS.

In [ ]:
!pip install faiss-cpu
!pip install chromadb
!pip install -U langchain-community
import os
import requests
from dotenv import load_dotenv
import openai
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings
from langchain.vectorstores import FAISS, Chroma
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import numpy as np

Explanation:

langchain: Framework for building LLM applications

faiss-cpu: Facebook's AI Similarity Search for vector databases (CPU version)

sentence-transformers: Library for sentence embeddings

chromadb: Vector database for storing embeddings

tiktoken: OpenAI's tokenizer for counting tokens

transformers: HuggingFace library for pre-trained models

torch: PyTorch for deep learning

accelerate: Library for optimizing model training/inference

# ## 3. Data Preparation

In [6]:


# %%
# Sample document - you can replace this with your own PDF or text files
def download_sample_pdf():
    url = "https://arxiv.org/pdf/2305.15334.pdf"  # Sample research paper
    response = requests.get(url)
    with open("sample.pdf", "wb") as f:
        f.write(response.content)
    return "sample.pdf"

# Download sample PDF
pdf_path = download_sample_pdf()
print(f"Downloaded PDF: {pdf_path}")

Downloaded PDF: sample.pdf


In [9]:
# Load and split documents
!pip install pypdf
import pypdf

def load_and_chunk_documents(file_path, chunk_size=1000, chunk_overlap=200):
    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)

    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks from {len(documents)} documents")
    return chunks

chunks = load_and_chunk_documents(pdf_path)
print(f"First chunk: {chunks[0].page_content[:200]}...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 4.0 MB/s eta 0:00:00
Created 80 chunks from 18 documents
First chunk: Gorilla: Large Language Model Connected with
Massive APIs
Shishir G. Patil1∗ Tianjun Zhang1,∗ Xin Wang2 Joseph E. Gonzalez1
1UC Berkeley 2Microsoft Research
sgp@berkeley.edu
Abstract
Large Language Mo...


In [11]:
print(f"Second chunk: {chunks[1].page_content[:200]}...")

Second chunk: demonstrates a strong capability to adapt to test-time document changes, enabling
flexible user updates or version changes. It also substantially mitigates the issue of
hallucination, commonly encount...


In [14]:
# ## 4. Embedding and Vector Store

# %%
# Initialize embeddings
def setup_vector_store(chunks, use_openai=True):
    if use_openai and openai.api_key:
        embeddings = OpenAIEmbeddings()
    else:
        # Use free embeddings model
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

    # Create vector store
    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store, embeddings

vector_store, embeddings = setup_vector_store(chunks, use_openai=False)


In [15]:
# %%
# Test similarity search
query = "What is retrieval augmented generation?"
similar_docs = vector_store.similarity_search(query, k=3)
print("Similar documents found:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:300] + "...")

Similar documents found:

--- Document 1 ---
Anthropic, renowned for its lengthy context capabilities; LLaMA-7B, a large language model by
Meta and the finest open-source model to date.
Retrievers The term Zero-shot (abbreviated as 0-shot in tables) refers to scenarios where no
retriever is used. The sole input to the model is the user’s natur...

--- Document 2 ---
Retriever-Aware training For training with retriever, the instruction-tuned dataset, also has an ad-
ditional "Use this API documentation for reference: <retrieved_API_doc_JSON>"
appended to the user prompt. Through this, we aim to teach the LLM to parse the second half
of the question to answer the...

--- Document 3 ---
of programming languages. arXiv preprint arXiv:2006.03511.
[21] Lazaridou, A., Gribovskaya, E., Stokowiec, W., and Grigorev, N. (2022). Internet-augmented
language models through few-shot prompting for open-domain question answering. arXiv preprint
arXiv:2203.05115.
[22] Li, R., Allal, L. B., Zi, Y ...


In [16]:
# %%
# Test similarity search
query = "What is retrieval augmented generation?"
similar_docs = vector_store.similarity_search(query, k=5)
print("Similar documents found:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:300] + "...")

Similar documents found:

--- Document 1 ---
Anthropic, renowned for its lengthy context capabilities; LLaMA-7B, a large language model by
Meta and the finest open-source model to date.
Retrievers The term Zero-shot (abbreviated as 0-shot in tables) refers to scenarios where no
retriever is used. The sole input to the model is the user’s natur...

--- Document 2 ---
Retriever-Aware training For training with retriever, the instruction-tuned dataset, also has an ad-
ditional "Use this API documentation for reference: <retrieved_API_doc_JSON>"
appended to the user prompt. Through this, we aim to teach the LLM to parse the second half
of the question to answer the...

--- Document 3 ---
of programming languages. arXiv preprint arXiv:2006.03511.
[21] Lazaridou, A., Gribovskaya, E., Stokowiec, W., and Grigorev, N. (2022). Internet-augmented
language models through few-shot prompting for open-domain question answering. arXiv preprint
arXiv:2203.05115.
[22] Li, R., Allal, L. B., Zi, Y ...

--


# ## 5. Retriever Setup

In [19]:
# Create different retriever types
def create_retrievers(vector_store):
    # Basic retriever
    basic_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # MMR retriever for diversity
    mmr_retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    )

    return {
        "basic": basic_retriever,
        "mmr": mmr_retriever
    }

retrievers = create_retrievers(vector_store)

# ## 6. Generator Setup

In [21]:
from langchain.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

In [22]:
def setup_free_llm():
    """Setup a free LLM for demonstration"""

    try:
        # Try to use a small, fast model
        print("Loading free LLM...")

        # Option 1: Use GPT2 (small and fast)
        model_name = "gpt2"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)

        # Add padding token if it doesn't exist
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        hf_pipeline = HuggingFacePipeline(pipeline=pipe)
        print("✅ Free LLM loaded successfully")
        return hf_pipeline

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("🔄 Using mock LLM instead...")
        return setup_mock_llm()

def setup_mock_llm():
    """Mock LLM for demonstration when real models fail"""
    from langchain.llms.fake import FakeListLLM

    responses = [
        "Based on the retrieved context, Retrieval-Augmented Generation (RAG) combines information retrieval with language generation.",
        "The document discusses how RAG enhances language models by incorporating external knowledge sources.",
        "According to the context, RAG helps reduce hallucinations in language models.",
        "The retrieved information suggests this is about improving AI response accuracy.",
        "RAG systems typically consist of a retriever component and a generator component."
    ]
    return FakeListLLM(responses=responses)

llm = setup_free_llm()

Loading free LLM...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu


✅ Free LLM loaded successfully


/tmp/ipython-input-1495048098.py:27: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  hf_pipeline = HuggingFacePipeline(pipeline=pipe)


# Custom prompt template

In [24]:
prompt_template = """Use the following context to answer the question.
If the context doesn't contain the answer, say you don't know.

Context: {context}

Question: {question}
Answer: """

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)
# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})


In [25]:
# Create RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print("✅ RAG chain created successfully")

✅ RAG chain created successfully



# ## 6. Test the RAG System
---



In [26]:
test_queries = [
    "What is retrieval augmented generation?",
    "How does RAG work?",
    "What are the benefits of RAG?",
    "What is the main topic of this document?"
]


In [27]:
def test_rag_system(chain, queries):
    print("🧪 Testing RAG System...")
    print("=" * 50)

    for i, query in enumerate(queries, 1):
        print(f"\nQuery {i}: {query}")
        print("-" * 30)

        try:
            result = chain({"query": query})
            print(f"Answer: {result['result']}")
            print(f"Sources: {len(result['source_documents'])} documents")

        except Exception as e:
            print(f"Error: {e}")
            # Fallback: simple similarity search
            docs = vector_store.similarity_search(query, k=1)
            if docs:
                print(f"Fallback answer: This document discusses: {docs[0].page_content[:200]}...")
            else:
                print("No relevant documents found.")

test_rag_system(rag_chain, test_queries)



🧪 Testing RAG System...

Query 1: What is retrieval augmented generation?
------------------------------


/tmp/ipython-input-2128259342.py:10: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chain({"query": query})


Answer: Use the following context to answer the question. 
If the context doesn't contain the answer, say you don't know.

Context: Anthropic, renowned for its lengthy context capabilities; LLaMA-7B, a large language model by
Meta and the finest open-source model to date.
Retrievers The term Zero-shot (abbreviated as 0-shot in tables) refers to scenarios where no
retriever is used. The sole input to the model is the user’s natural language prompt. For BM25, we
consider each API as a separate document. During retrieval, we use the user’s query to search the
index and fetch the most relevant (top-1) API. This API is concatenated with the user’s prompt
to query the LLMs. Similarly, GPT-Index refers to the retrieval model text-davinci-003 from
OpenAI. Like BM25, each API call is indexed as an individual document, and the most relevant
document, given a user query, is retrieved and appended to the user prompt. Lastly, we include
an Oracle retriever, which serves two purposes: first, to iden

# ## 7. Simple PDF Content Viewer

In [29]:
!pip install pdfplumber
import pdfplumber

def show_pdf_content(pdf_path, pages_to_show=2):
    """Show what's in the PDF"""
    print("\n📄 PDF CONTENT PREVIEW")
    print("=" * 50)

    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total pages: {len(pdf.pages)}")

        for i in range(min(pages_to_show, len(pdf.pages))):
            page = pdf.pages[i]
            text = page.extract_text()

            print(f"\n📍 PAGE {i+1}:")
            print("-" * 40)
            if text:
                # Show first 500 characters
                preview = text[:500] + "..." if len(text) > 500 else text
                print(preview)
            else:
                print("No extractable text")

show_pdf_content(pdf_path)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 86.5 MB/s eta 0:00:00

📄 PDF CONTENT PREVIEW
Total pages: 18

📍 PAGE 1:
----------------------------------------
Gorilla: Large Language Model Connected with
Massive APIs
ShishirG.Patil1∗ TianjunZhang1,∗ XinWang2 JosephE.Gonzalez1
1UCBerkeley 2MicrosoftResearch
sgp@berkeley.edu
Abstract
LargeLanguageModels(LLMs)haveseenanimpressivewaveofadvancesre-
cently, with models now excelling in a variety of tasks, such as mathematical
reasoningandprogramsynthesis. However,theirpotentialtoeffectivelyusetools
viaAPIcallsremainsunfulfilled.Thisisachallengingtaskevenfortoday’sstate-of-
the-artLLMssuchasGPT-4,largelyduet...

📍 PAGE 2:
--